# 01 Data Understanding and Cleaning

UCI Bank Marketing Dataset — `bank-additional-full.csv`

This notebook covers **data understanding and cleaning only**. No encoding, scaling, train/test splitting, feature engineering, or modeling happens here. The `duration` feature is identified and described, but its leakage implications are deferred to `03_leakage_investigation.ipynb`.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

## 2. Load Dataset

The file is **semicolon-separated**, not comma-separated. Loading it with the default separator would not raise an error — it would silently return a single-column DataFrame where every row is one long string. `sep=";"` is required to load it correctly.

In [2]:
df = pd.read_csv("../data/raw/bank-additional-full.csv", sep=";")
df.shape

(41188, 21)

## 3. Initial Inspection

In [3]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [4]:
df.tail()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,334,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,383,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,189,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,442,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41187,74,retired,married,professional.course,no,yes,no,cellular,nov,fri,239,3,999,1,failure,-1.1,94.767,-50.8,1.028,4963.6,no


In [5]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'housing',
 'loan',
 'contact',
 'month',
 'day_of_week',
 'duration',
 'campaign',
 'pdays',
 'previous',
 'poutcome',
 'emp.var.rate',
 'cons.price.idx',
 'cons.conf.idx',
 'euribor3m',
 'nr.employed',
 'y']

In [6]:
df.index

RangeIndex(start=0, stop=41188, step=1)

## 4. Dataset Structure

In [7]:
df.dtypes

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

In [9]:
df.describe()

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
count,41188.00000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000,41188.000000
mean,40.02406,258.285010,2.567593,962.475454,0.172963,0.081886,93.575664,-40.502600,3.621291,5167.035911
std,10.42125,259.279249,2.770014,186.910907,0.494901,1.570960,0.578840,4.628198,1.734447,72.251528
min,17.00000,0.000000,1.000000,0.000000,0.000000,-3.400000,92.201000,-50.800000,0.634000,4963.600000
25%,32.00000,102.000000,1.000000,999.000000,0.000000,-1.800000,93.075000,-42.700000,1.344000,5099.100000
50%,38.00000,180.000000,2.000000,999.000000,0.000000,1.100000,93.749000,-41.800000,4.857000,5191.000000
75%,47.00000,319.000000,3.000000,999.000000,0.000000,1.400000,93.994000,-36.400000,4.961000,5228.100000
max,98.00000,4918.000000,56.000000,999.000000,7.000000,1.400000,94.767000,-26.900000,5.045000,5228.100000


In [10]:
df.describe(include='object')

,job,marital,education,default,housing,loan,contact,month,day_of_week,poutcome,y
count,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188,41188
unique,12,4,8,3,3,3,2,10,5,3,2
top,admin.,married,university.degree,no,yes,no,cellular,may,thu,nonexistent,no
freq,10422,24928,12168,32588,21576,33950,26144,13769,8623,35563,36548


## 5. Feature Understanding

Column reference (from the UCI dataset documentation):

**Bank client data**
- `age` — client age (numeric)
- `job` — type of job (categorical)
- `marital` — marital status (categorical)
- `education` — education level (categorical)
- `default` — has credit in default? (categorical: no/yes/unknown)
- `housing` — has a housing loan? (categorical)
- `loan` — has a personal loan? (categorical)

**Last contact of the current campaign**
- `contact` — contact communication type (cellular/telephone)
- `month` — last contact month
- `day_of_week` — last contact day of week
- `duration` — last contact duration, in seconds (numeric) — flagged for the dedicated leakage investigation in notebook 03

**Other campaign attributes**
- `campaign` — number of contacts performed during this campaign, for this client
- `pdays` — days since the client was last contacted in a previous campaign (`999` means never previously contacted — a sentinel, not a real day count)
- `previous` — number of contacts performed before this campaign, for this client
- `poutcome` — outcome of the previous marketing campaign

**Social and economic context (external indicators)**
- `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`, `nr.employed`

**Target**
- `y` — did the client subscribe to a term deposit? (yes/no)

In [11]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Numerical ({len(numerical_cols)}):', numerical_cols)
print(f'Categorical ({len(categorical_cols)}):', categorical_cols)

Numerical (10): ['age', 'duration', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical (11): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome', 'y']


In [12]:
df[categorical_cols].nunique().sort_values()

contact         2
y               2
default         3
housing         3
loan            3
poutcome        3
marital         4
day_of_week     5
education       8
month          10
job            12
dtype: int64

In [13]:
for col in categorical_cols:
    print(col, '->', df[col].unique())
    print()

job -> ['housemaid' 'services' 'admin.' 'blue-collar' 'technician' 'retired'
 'management' 'unemployed' 'self-employed' 'unknown' 'entrepreneur'
 'student']

marital -> ['married' 'single' 'divorced' 'unknown']

education -> ['basic.4y' 'high.school' 'basic.6y' 'basic.9y' 'professional.course'
 'unknown' 'university.degree' 'illiterate']

default -> ['no' 'unknown' 'yes']

housing -> ['no' 'yes' 'unknown']

loan -> ['no' 'yes' 'unknown']

contact -> ['telephone' 'cellular']

month -> ['may' 'jun' 'jul' 'aug' 'oct' 'nov' 'dec' 'mar' 'apr' 'sep']

day_of_week -> ['mon' 'tue' 'wed' 'thu' 'fri']

poutcome -> ['nonexistent' 'failure' 'success']

y -> ['no' 'yes']



## 6. Data Quality Checks

Before deciding what to clean, we look at value ranges and known sentinel values so we can distinguish genuinely suspicious data from values that only *look* unusual out of context.

In [14]:
df[numerical_cols].agg(['min', 'max'])

,age,duration,campaign,pdays,previous,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
min,17,0,1,0,0,-3.4,92.201,-50.8,0.634,4963.6
max,98,4918,56,999,7,1.4,94.767,-26.9,5.045,5228.1


`age` (17–98) and `campaign` (1–56) are wide but plausible — not impossible values, just skewed. No action needed here.

In [15]:
df['pdays'].value_counts().head()

pdays
999    39673
3        439
6        412
4        118
9         64
Name: count, dtype: int64

`999` dominates `pdays` — it's a placeholder meaning *"never contacted before"*, not a real day count. Treating it as a normal number would badly distort any statistics on this column. Noted for feature engineering (e.g. a `was_previously_contacted` flag); not transformed here.

In [16]:
df.loc[df['duration'] == 0, ['duration', 'y']]

,duration,y
6251,0,no
23031,0,no
28063,0,no
33015,0,no


All 4 rows with `duration == 0` have `y == 'no'` — consistent with the UCI documentation: a call that never connects cannot end in a subscription. These are valid records, not data errors, so they are kept. Any further decision about `duration` itself belongs to notebook 03.

In [17]:
df['default'].value_counts()

default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64

Only 3 rows have `default == 'yes'` — extremely rare, but not impossible for a real bank population. Kept as-is.

In [18]:
pd.crosstab(df['previous'] == 0, df['poutcome'])

poutcome,failure,nonexistent,success
previous,,,
False,4252,0,1373
True,0,35563,0


`previous == 0` lines up exactly with `poutcome == 'nonexistent'` — the two columns are internally consistent, no contradiction found.

## 7. Missing and Unknown Values

In [19]:
df.isnull().sum()

age               0
job               0
marital           0
education         0
default           0
housing           0
loan              0
contact           0
month             0
day_of_week       0
duration          0
campaign          0
pdays             0
previous          0
poutcome          0
emp.var.rate      0
cons.price.idx    0
cons.conf.idx     0
euribor3m         0
nr.employed       0
y                 0
dtype: int64

In [20]:
df.isnull().sum().sum()

np.int64(0)

No true `NaN` values anywhere in the dataset. However, several categorical columns encode missingness as the literal string `'unknown'`, which `isnull()` does not detect — checked explicitly below.

In [21]:
unknown_counts = pd.DataFrame({
    'unknown_count': (df[categorical_cols] == 'unknown').sum(),
    'unknown_pct': ((df[categorical_cols] == 'unknown').sum() / len(df) * 100).round(2)
})
unknown_counts[unknown_counts['unknown_count'] > 0].sort_values('unknown_count', ascending=False)

,unknown_count,unknown_pct
default,8597,20.87
education,1731,4.20
housing,990,2.40
loan,990,2.40
job,330,0.80
marital,80,0.19


`default` has ~20.9% `unknown` — far too large a share to drop or naively impute without losing a substantial part of the dataset. `education` (~4.2%), `housing`/`loan` (~2.4%), `job` (~0.8%), and `marital` (~0.2%) are smaller but still present.

`unknown` is unlikely to be missing at random — clients who don't disclose whether they're in credit default may differ systematically from those who do, which means the missingness itself could be informative. For that reason, **`unknown` is kept as an explicit category in every column, not dropped or imputed**. Whether to encode it specially is a feature engineering decision for a later notebook, not a cleaning decision made here.

## 8. Duplicate Analysis

In [22]:
df.duplicated().sum()

np.int64(12)

In [23]:
df[df.duplicated(keep=False)].sort_values(numerical_cols).head(12)

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
28476,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.8,93.075,-47.1,1.423,5099.1,no
28477,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.8,93.075,-47.1,1.423,5099.1,no
14155,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no
14234,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no
18464,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.4,93.918,-42.7,4.968,5228.1,no
18465,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.4,93.918,-42.7,4.968,5228.1,no
32505,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
32516,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
12260,36,retired,married,unknown,no,no,no,telephone,jul,thu,88,1,999,0,nonexistent,1.4,93.918,-42.7,4.966,5228.1,no
12261,36,retired,married,unknown,no,no,no,telephone,jul,thu,88,1,999,0,nonexistent,1.4,93.918,-42.7,4.966,5228.1,no


12 rows are exact duplicates across all 21 columns, including `duration` — a continuous value measured in seconds. There is no customer ID in this dataset, so duplicates can't be confirmed against an identifier, but two independent customers coincidentally matching on every single field, including an exact contact duration, is implausible. These are treated as accidental duplicate records (e.g. export/logging artifacts) and removed in the cleaning step below.

## 9. Target Variable Analysis

In [24]:
df['y'].value_counts()

y
no     36548
yes     4640
Name: count, dtype: int64

In [25]:
(df['y'].value_counts(normalize=True) * 100).round(2)

y
no     88.73
yes    11.27
Name: proportion, dtype: float64

~88.7% `no` vs ~11.3% `yes` — a substantial class imbalance. This is not corrected here: resampling, class weighting, and metric choice (precision/recall/PR-AUC over accuracy) are modeling-phase decisions. It's documented now so it isn't forgotten later.

## 10. Data Cleaning

Decisions made, each tied to a finding above:

1. **Drop the 12 exact duplicate rows** — identical across all 21 fields including a continuous `duration` value; not plausible as independent records.

Decisions explicitly **not** made in this notebook, and why:

- **`unknown` values are kept**, in every column — too large a share (up to ~20.9%) to drop, and likely not missing-at-random, so dropping or imputing now would be a guess rather than a justified decision.
- **`pdays == 999` is left untouched** — it's a sentinel, not a real value; recoding it is feature engineering, not cleaning.
- **`duration` is left untouched** — its treatment depends on the prediction-scenario leakage analysis in notebook 03, not on data quality.
- **No dtype conversions, encoding, or scaling** — out of scope for this phase.

The raw DataFrame `df` is not modified in place; a new `df_clean` is created instead so `data/raw/bank-additional-full.csv` stays reproducible from source.

In [26]:
df_clean = df.drop_duplicates().reset_index(drop=True)
df_clean.shape

(41176, 21)

## 11. Post-Cleaning Validation

In [27]:
df_clean.duplicated().sum()

np.int64(0)

In [28]:
df_clean.isnull().sum().sum()

np.int64(0)

In [29]:
(df_clean['y'].value_counts(normalize=True) * 100).round(2)

y
no     88.73
yes    11.27
Name: proportion, dtype: float64

In [30]:
df_clean.dtypes

age                 int64
job                object
marital            object
education          object
default            object
housing            object
loan               object
contact            object
month              object
day_of_week        object
duration            int64
campaign            int64
pdays               int64
previous            int64
poutcome           object
emp.var.rate      float64
cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
y                  object
dtype: object

Shape, duplicate count, null count, target distribution, and dtypes all confirmed after cleaning. Only the duplicate rows changed — everything else matches the raw data exactly.

In [31]:
df_clean.to_csv('../data/processed/bank_marketing_cleaned.csv', index=False)

Saved to `data/processed/bank_marketing_cleaned.csv`. This is the dataset later notebooks (EDA onward) should load — it removes only the 12 confirmed accidental duplicates and changes nothing else, so every later decision (leakage, encoding, imputation strategy) still starts from a fully documented, unaltered-in-substance state.

## 12. Summary and Key Findings

**Discovered**
- 41,188 rows, 21 columns; no true `NaN` values anywhere.
- Six categorical columns contain `'unknown'` as a placeholder: `default` (~20.9%), `education` (~4.2%), `housing`/`loan` (~2.4% each), `job` (~0.8%), `marital` (~0.2%).
- 12 exact full-row duplicates.
- `pdays` is dominated by the sentinel value `999` ("never previously contacted").
- `duration == 0` occurs in 4 rows, always paired with `y == 'no'` — logically consistent, not an error.
- `previous` and `poutcome` are internally consistent with each other.
- Target is imbalanced: ~11.3% `yes`, ~88.7% `no`.

**Cleaned**
- Removed 12 exact duplicate rows. Nothing else was altered.

**Intentionally not changed**
- `'unknown'` values in all six affected columns — kept as an explicit category.
- `pdays == 999` sentinel — left as-is, flagged for feature engineering.
- `duration` — left as-is; leakage decision deferred to notebook 03.
- No encoding, scaling, splitting, or imputation performed.

**Carry forward to later notebooks**
- `duration` needs a full leakage investigation against the defined prediction scenario (notebook 03) before any modeling decision is made about it.
- `pdays == 999` likely warrants a derived binary flag (e.g. `was_previously_contacted`) during feature engineering, rather than using the raw sentinel value numerically.
- The `'unknown'` category's encoding strategy (own category vs. imputation) should be decided during feature engineering, informed by EDA on whether `'unknown'` correlates with the target.
- Class imbalance (~11.3% positive) must inform metric choice and model strategy — accuracy alone will be misleading.
- All downstream notebooks should load `data/processed/bank_marketing_cleaned.csv`, not the raw file.